# 06 · Nebraska replication of the drought-penalty method (pre-registered)

**Question.** Does the notebook 05 drought-penalty method, designed on Iowa, work **unchanged** on Nebraska's mostly-dryland counties?

**Status.** This cell is the full plan, written and committed **before any Nebraska result exists**. Every rule below is fixed in advance. Nothing is tuned after seeing results. Whatever the outcome, it is reported.

Plan first committed locally as 4be926d (8:59 pm), then re-committed as 2bc1c15 with a corrected author email (same content) and pushed at 9:22 pm; the feasibility gate ran in between. No outcome was computed because both crops stopped at the gate.

## Method (imported, not copied)
- All logic is imported from `soil_planner.soil_tool`: `compute_penalties`, `penalty_ranking_results`, `ranking_summary_table`, and the parameters `WINDOW`, `MIN_NORMAL_YEARS`, `SEV_D2`, `MIN_D2_YEARS_FOR_SCORE`, `TRAIN_YEARS`, `TEST_YEARS`, `N_BOOT`, `BOOT_SEED`, `CI`, `TOP_SHARE`, and the placebo definitions `REAL_TEST`, `PLACEBO_ZERO`, `PLACEBO_ALL`. `soil_tool.py` is not edited.
- `soil_tool.load_iowa()` is hard-wired to Iowa, so this notebook builds the Nebraska input itself from `src.data_loader.load_master()`: rows with `state_alpha == "NE"`, `crop` in `{CORN, SOYBEANS}`, a reported yield (`yield_per_acre` not null), `fips` kept as 5-character text, and passes them through the **same** functions. Corn and soybeans are run separately, as in Iowa.
- **Fixed parameters, identical to Iowa** (read from the module and asserted at run time): `WINDOW = 3` years around the target year for the county's normal; at least `MIN_NORMAL_YEARS = 2` normal years; drought year = `max_drought_severity >= SEV_D2 = 3` (D2+); at least `MIN_D2_YEARS_FOR_SCORE = 3` drought years with a penalty for a score; TRAIN `2000–2015`, TEST `2016–2025`; `N_BOOT = 5000` bootstrap draws with `BOOT_SEED = 20261017`; 95% range from the 2.5th and 97.5th percentiles; top third = `TOP_SHARE = 1/3`; the same three scores (penalty score, drought frequency, soil water storage) and the same two placebos (zero-drought test years; non-D2+ counties in all test years).
- **One deliberate difference, fixed now:** the minimum number of counties for a usable year is **10** instead of Iowa's 15, only because the dryland pool is smaller. It is applied by setting the module attribute `soil_tool.MIN_COUNTIES_PER_YEAR = 10` at run time for the Nebraska run (the file is unchanged; the Iowa comparison is computed first, with its own 15, and the attribute is restored afterwards). That attribute governs both which years count as usable and the per-year evaluation inside `penalty_ranking_results`.
- **"Remove what hit everyone" over dryland counties only.** `compute_penalties` subtracts the statewide median log yield of the rows it receives. Because only Nebraska dryland rows are passed, that median is the median over the dryland counties, as required.

## Dryland definition (fixed)
- From `src.data_loader.load_irrigation()`: Nebraska rows for the crop; for each county, the **mean of `irrigated_share` over the years with a value**. A county is **dryland** if that mean is **below 0.20**.
- Counties with **no irrigation rows for that crop are excluded**, not assumed dryland.
- The irrigation extension mostly ends in 2018. Dryland status is therefore measured on the years available and **assumed to hold afterwards**; the notebook reports the last Nebraska year with data.

## Step 1: feasibility gate (reported before anything else is computed)
Per crop: (a) number of dryland counties; (b) number of dryland counties with a reported yield in **most years, defined as at least 14 of the 26 years**; (c) number of TRAIN years and of TEST years with **at least 10 dryland counties at D2+ with a reported yield**.

**Proceed to Step 2 for a crop only if (a) ≥ 20 and the TEST count in (c) ≥ 3.** Otherwise stop for that crop and report why. Note that (c) counts D2+ counties with a reported yield; the test itself additionally needs a valid penalty for each county-year, so a year that passes the gate can still be dropped inside the test. That is accepted in advance.

## Step 2: replication (crops that pass)
- `compute_penalties(nebraska_dryland_rows, split=True)` (normals never cross 2015/2016), then `penalty_ranking_results(...)` with the threshold set to 10 as above: TRAIN-only scores tested on TEST drought years; outcomes are the **Spearman rank correlation** (signed so positive = correct ordering, chance 0) and the **top-third hit rate** (chance 0.33), each averaged over usable test years with a year-bootstrap 95% range; plus the two placebos.
- Results are shown **side by side with Iowa's** (same functions, Iowa's threshold of 15), with the number of years and of counties each rests on.

## Pre-registered success rule
The method **succeeds** for a crop if, for the **penalty score** on the real test, **the 95% range of the top-third hit rate lies entirely above 0.33 AND the broad placebo (non-D2+ counties, all test years) stays at chance**, meaning its 95% ranges for the penalty score include the chance value for both metrics (0 for Spearman, 0.33 for the top-third hit rate). Any other pattern is reported as **not meeting the rule**. The Spearman result, drought frequency and storage are reported but do not enter the rule.

## Step 3: plain-language summary (written after the results, judged by the rule above)
Did the method meet the rule? Did the placebo stay clean? How much weight can the result bear given the number of years? State that irrigation data mostly ends in 2018 so later dryland status is assumed, not measured. If a range clears chance on few years, note that the size of the effect is likely overstated (Gelman, Hill & Vehtari, *Regression and Other Stories*, §4.4, §16.1). Do not state conclusions the data does not support.

## Scope
Nebraska is **not** added to the app, the priority map or `soil_tool.py`. Nothing under `data/` or `src/` is modified. The notebook runs from the repo root with `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=600 notebooks/06_nebraska_replication.ipynb`.


## Setup: import the method, assert the fixed parameters

In [1]:
import sys
sys.path.insert(0, "..")
import contextlib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

from soil_planner import soil_tool as st          # the method; not copied, not modified
from src.data_loader import load_master, load_irrigation

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 60)

# ---- fixed parameters, pre-registered; asserted against the module so nothing can drift ----
assert st.WINDOW == 3 and st.MIN_NORMAL_YEARS == 2 and st.SEV_D2 == 3 and st.MIN_D2_YEARS_FOR_SCORE == 3
assert st.TRAIN_YEARS == range(2000, 2016) and st.TEST_YEARS == range(2016, 2026)
assert st.N_BOOT == 5000 and st.BOOT_SEED == 20261017 and tuple(st.CI) == (2.5, 97.5) and abs(st.TOP_SHARE - 1 / 3) < 1e-12
IOWA_MIN_COUNTIES = st.MIN_COUNTIES_PER_YEAR          # 15, Iowa's threshold, used for the Iowa comparison
assert IOWA_MIN_COUNTIES == 15
NE_MIN_COUNTIES = 10            # pre-registered replacement for the smaller dryland pool (test and gate)
DRYLAND_MAX_SHARE = 0.20        # dryland = mean irrigated share below this
MOST_YEARS = 14                 # "reported in most years" = at least 14 of 26
GATE_MIN_DRYLAND = 20           # proceed only with >= 20 dryland counties ...
GATE_MIN_TEST_YEARS = 3         # ... and >= 3 TEST years with >= NE_MIN_COUNTIES D2+ dryland counties
CROPS = dict(st.CROPS)          # {"Corn": "CORN", "Soybeans": "SOYBEANS"}
FIPS, YEAR, CROP, SEV, YIELD = st.COL_FIPS, st.COL_YEAR, st.COL_CROP, st.COL_SEV, st.COL_YIELD

@contextlib.contextmanager
def module_settings(**overrides):
    """Thin wrapper: temporarily set soil_tool module attributes (MIN_COUNTIES_PER_YEAR, CROPS) at run time,
    then restore them. The file soil_planner/soil_tool.py is not modified."""
    saved = {k: getattr(st, k) for k in overrides}
    try:
        for k, v in overrides.items():
            setattr(st, k, v)
        yield
    finally:
        for k, v in saved.items():
            setattr(st, k, v)

print(f"WINDOW={st.WINDOW}, MIN_NORMAL_YEARS={st.MIN_NORMAL_YEARS}, D2 code={st.SEV_D2}, min drought years for a score={st.MIN_D2_YEARS_FOR_SCORE}, "
      f"TRAIN={st.TRAIN_YEARS.start}-{st.TRAIN_YEARS.stop-1}, TEST={st.TEST_YEARS.start}-{st.TEST_YEARS.stop-1}, "
      f"N_BOOT={st.N_BOOT}, seed={st.BOOT_SEED}, CI={st.CI}, top share={st.TOP_SHARE:.3f}")
print(f"Iowa min counties per usable year = {IOWA_MIN_COUNTIES}; Nebraska (pre-registered) = {NE_MIN_COUNTIES}; dryland = mean irrigated share < {DRYLAND_MAX_SHARE}")

WINDOW=3, MIN_NORMAL_YEARS=2, D2 code=3, min drought years for a score=3, TRAIN=2000-2015, TEST=2016-2025, N_BOOT=5000, seed=20261017, CI=(2.5, 97.5), top share=0.333
Iowa min counties per usable year = 15; Nebraska (pre-registered) = 10; dryland = mean irrigated share < 0.2


## Dryland definition

Mean `irrigated_share` over the years with a value, per Nebraska county and crop. Dryland = below 0.20. Counties without irrigation rows for the crop are excluded.

In [2]:
ir = load_irrigation()
assert ir[FIPS].map(type).eq(str).all() and ir[FIPS].str.len().eq(5).all()
ne_ir = ir[(ir["state_alpha"] == "NE") & ir[CROP].isin(CROPS.values()) & ir["irrigated_share"].notna()]
share = (ne_ir.groupby([CROP, FIPS])
         .agg(mean_irrigated_share=("irrigated_share", "mean"), years_with_data=(YEAR, "nunique"),
              first_year=(YEAR, "min"), last_year=(YEAR, "max")).reset_index())
share["dryland"] = share["mean_irrigated_share"] < DRYLAND_MAX_SHARE
DRYLAND = {crop: sorted(share.loc[(share[CROP] == crop) & share["dryland"], FIPS]) for crop in CROPS.values()}

summary = (share.groupby(CROP).agg(counties_with_irrigation_data=(FIPS, "nunique"), dryland_counties=("dryland", "sum"),
                                   median_irrigated_share=("mean_irrigated_share", "median"),
                                   last_year_with_data=("last_year", "max"))
           .assign(dryland_counties=lambda d: d["dryland_counties"].astype(int)))
display(summary)
print("Years with Nebraska irrigation data, by number of county-crop rows:")
print(ne_ir.groupby(YEAR).size().to_string())

,counties_with_irrigation_data,dryland_counties,median_irrigated_share,last_year_with_data
crop,,,,
CORN,91,13,0.700611,2018
SOYBEANS,80,18,0.719000,2018


Years with Nebraska irrigation data, by number of county-crop rows:
year
2000    166
2001    168
2002    166
2003    162
2004    169
2005    167
2006    167
2007    155
2008    124
2009    131
2010    127
2011    129
2012    120
2013    111
2014     91
2015     82
2016     88
2017     71
2018     45


## Step 1: feasibility gate

Per crop: dryland counties; dryland counties with a reported yield in at least 14 of 26 years; TRAIN and TEST years with at least 10 dryland counties at D2+ with a reported yield. **Proceed only if dryland counties ≥ 20 and TEST years ≥ 3.** Decided by the rule, not by hand.

In [3]:
m = load_master()
ne = m[(m["state_alpha"] == "NE") & m[CROP].isin(CROPS.values()) & m[YIELD].notna()].copy()
assert ne[FIPS].map(type).eq(str).all() and ne[FIPS].str.len().eq(5).all() and (ne[YIELD] > 0).all()
assert ne[SEV].notna().all()

gate_rows, PASS = [], {}
for crop_label, crop in CROPS.items():
    d = ne[(ne[CROP] == crop) & ne[FIPS].isin(DRYLAND[crop])]
    yrs = d.groupby(FIPS)[YEAR].nunique()
    d2 = d[d[SEV] >= st.SEV_D2].groupby(YEAR)[FIPS].nunique()
    train_ok = [int(y) for y in d2[d2 >= NE_MIN_COUNTIES].index if y in st.TRAIN_YEARS]
    test_ok = [int(y) for y in d2[d2 >= NE_MIN_COUNTIES].index if y in st.TEST_YEARS]
    PASS[crop] = (len(DRYLAND[crop]) >= GATE_MIN_DRYLAND) and (len(test_ok) >= GATE_MIN_TEST_YEARS)
    gate_rows.append({"crop": crop_label, "dryland counties": len(DRYLAND[crop]),
                      f"with yield in >= {MOST_YEARS} of 26 years": int((yrs >= MOST_YEARS).sum()),
                      f"TRAIN years with >= {NE_MIN_COUNTIES} D2+ dryland counties": f"{len(train_ok)}: {train_ok}",
                      f"TEST years with >= {NE_MIN_COUNTIES} D2+ dryland counties": f"{len(test_ok)}: {test_ok}",
                      "gate": "PASS -> Step 2" if PASS[crop] else "STOP"})
    print(f"{crop_label}: D2+ dryland counties with a reported yield, by year:")
    print(d2.reindex(range(2000, 2026), fill_value=0).to_frame("counties").T.to_string())
gate = pd.DataFrame(gate_rows).set_index("crop")
display(gate)
for crop_label, crop in CROPS.items():
    if not PASS[crop]:
        print(f"STOP for {crop_label}: rule requires >= {GATE_MIN_DRYLAND} dryland counties and >= {GATE_MIN_TEST_YEARS} TEST years with >= {NE_MIN_COUNTIES} D2+ dryland counties.")
PASSING = {k: v for k, v in CROPS.items() if PASS[v]}
print("Crops proceeding to Step 2:", list(PASSING) or "none")

Corn: D2+ dryland counties with a reported yield, by year:
year      2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
counties    12     0    13    11     1     0    10     1     0     0     0     0    13     7     0     0     0     1     2     0     4     3     9     9     0     2
Soybeans: D2+ dryland counties with a reported yield, by year:
year      2000  2001  2002  2003  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  2025
counties    17     0    18    16     1     0    15     1     0     0     0     0    18    12     2     0     0     2     2     0    10     3    16    13    12     3


,dryland counties,with yield in >= 14 of 26 years,TRAIN years with >= 10 D2+ dryland counties,TEST years with >= 10 D2+ dryland counties,gate
crop,,,,,
Corn,13,13,"5: [2000, 2002, 2003, 2006, 2012]",0: [],STOP
Soybeans,18,18,"6: [2000, 2002, 2003, 2006, 2012, 2013]","4: [2020, 2022, 2023, 2024]",STOP


STOP for Corn: rule requires >= 20 dryland counties and >= 3 TEST years with >= 10 D2+ dryland counties.
STOP for Soybeans: rule requires >= 20 dryland counties and >= 3 TEST years with >= 10 D2+ dryland counties.
Crops proceeding to Step 2: none


## Step 2: replication with the unchanged method

Iowa first (its own threshold of 15, `load_iowa_with_lon()` as in notebook 05). Then Nebraska dryland rows through the same `compute_penalties(split=True)` and `penalty_ranking_results`, with the module attribute `MIN_COUNTIES_PER_YEAR` set to 10 and `CROPS` restricted to the crops that passed the gate, both restored afterwards. The statewide median inside `compute_penalties` is taken over the rows passed, i.e. the dryland counties only.

In [4]:
# Iowa, exactly as in notebook 05
ia_split = st.compute_penalties(st.load_iowa_with_lon(), split=True)
rk_ia = st.penalty_ranking_results(ia_split)
assert st.MIN_COUNTIES_PER_YEAR == IOWA_MIN_COUNTIES

rk_ne, ne_split = None, None
if PASSING:
    ne_dry = pd.concat([ne[(ne[CROP] == crop) & ne[FIPS].isin(DRYLAND[crop])] for crop in PASSING.values()], ignore_index=True)
    with module_settings(MIN_COUNTIES_PER_YEAR=NE_MIN_COUNTIES, CROPS=PASSING):
        ne_split = st.compute_penalties(ne_dry, split=True)     # median over dryland counties only (only dryland rows are passed)
        rk_ne = st.penalty_ranking_results(ne_split)
    assert st.MIN_COUNTIES_PER_YEAR == IOWA_MIN_COUNTIES and st.CROPS == CROPS   # restored
    print(f"Nebraska dryland rows: {len(ne_dry):,} county-years, {ne_dry[FIPS].nunique()} counties; "
          f"penalty values: {int(ne_split[st.COL_PEN].notna().sum()):,}")
    print("Sanity, Nebraska: mean penalty by kind (placebo should sit near zero):")
    display(ne_split.groupby([CROP, st.COL_KIND])[st.COL_PEN].describe()[["count", "mean", "std", "50%"]].round(2))
    for crop_label, crop in PASSING.items():
        info = rk_ne["test_years"][crop]
        print(f"{crop_label} (Nebraska dryland): usable TEST years (>= {NE_MIN_COUNTIES} D2+ counties with a penalty): "
              + ", ".join(f"{y} ({n})" for y, n in info["usable"].items())
              + f" | zero-drought TEST years: {info['zero_drought']} | other: {info['other']}")
else:
    print("No crop passed the gate; Step 2 not run.")

No crop passed the gate; Step 2 not run.


### Results side by side

Mean [95% year-bootstrap range] of each metric for the real test and both placebos; `n_years` is the number of usable test years behind each estimate, and the usable-year county counts are listed above. Chance: 0 for Spearman, 0.33 for the top-third hit rate.

In [5]:
def side_by_side(crop):
    cols = ["test", "score", "metric", "mean [95% range]", "n_years", "years_above_chance", "range_above_chance"]
    ia = rk_ia["summary"][rk_ia["summary"]["crop"] == crop][cols].rename(columns=lambda c: c if c in ("test", "score", "metric") else f"Iowa: {c}")
    if rk_ne is None or crop not in rk_ne["summary"]["crop"].values:
        return ia.set_index(["test", "score", "metric"])
    nb_ = rk_ne["summary"][rk_ne["summary"]["crop"] == crop][cols].rename(columns=lambda c: c if c in ("test", "score", "metric") else f"Nebraska dryland: {c}")
    return ia.merge(nb_, on=["test", "score", "metric"], how="left").set_index(["test", "score", "metric"])

for crop_label, crop in CROPS.items():
    display(Markdown(f"**{crop_label}** — Iowa (99 counties, threshold 15) vs Nebraska dryland "
                     f"({len(DRYLAND[crop])} counties, threshold {NE_MIN_COUNTIES}){'' if PASS[crop] else ' — Nebraska stopped at the gate'}"))
    display(side_by_side(crop).loc[[st.REAL_TEST, st.PLACEBO_ZERO, st.PLACEBO_ALL]])

if rk_ne is not None:
    print("Per-year values, Nebraska dryland, real test (Spearman; positive = correct ordering) and top-third hit rate:")
    for crop_label, crop in PASSING.items():
        py, _ = rk_ne["results"][(crop, st.REAL_TEST)]
        t = pd.concat({n: p["spearman"] for n, p in py.items()}, axis=1)
        t.insert(0, "n_counties", py["penalty_score"]["n"].astype(int))
        t = t.join(pd.concat({n: p["top_third_hit"] for n, p in py.items()}, axis=1).add_prefix("hit: "))
        print(crop_label); display(t.round(2))

**Corn** — Iowa (99 counties, threshold 15) vs Nebraska dryland (13 counties, threshold 10) — Nebraska stopped at the gate

Iowa: mean [95% range]  Iowa: n_years  Iowa: years_above_chance  Iowa: range_above_chance
test                                      score         metric                                                                                                 
real: drought years                       penalty_score spearman        +0.22 [+0.13, +0.31]              5                         5                      True
                                                        top_third_hit   +0.45 [+0.36, +0.54]              5                         4                      True
                                          drought_freq  spearman        -0.14 [-0.40, +0.15]              6                         2                     False
                                                        top_third_hit   +0.31 [+0.14, +0.49]              6                         3                     False
                                          storage       spearman        +0.17 [+0.05, +0.29]              6                         6                      True
                                                        top_third_hit   +0.36 [+0.21, +0.46]              6                         4                     False
placebo: zero-drought years               penalty_score spearman        +0.17 [-0.25, +0.42]              3                         2                     False
                                                        top_third_hit   +0.40 [+0.18, +0.52]              3                         2                     False
                                          drought_freq  spearman        -0.05 [-0.19, +0.07]              3                         1                     False
                                                        top_third_hit   +0.34 [+0.26, +0.41]              3                         1                     False
                                          storage       spearman        +0.11 [+0.01, +0.30]              3                         3                      True
                                                        top_third_hit   +0.29 [+0.17, +0.37]              3                         2                     False
placebo: non-D2+ counties, all test years penalty_score spearman        +0.04 [-0.17, +0.27]              9                         4                     False
                                                        top_third_hit   +0.35 [+0.25, +0.46]              9                         4                     False
                                          drought_freq  spearman        -0.03 [-0.17, +0.12]              9                         4                     False
                                                        top_third_hit   +0.35 [+0.26, +0.43]              9                         5                     False
                                          storage       spearman        -0.01 [-0.14, +0.14]              9                         4                     False
                                                        top_third_hit   +0.30 [+0.25, +0.34]              9                         4                     False

**Soybeans** — Iowa (99 counties, threshold 15) vs Nebraska dryland (18 counties, threshold 10) — Nebraska stopped at the gate

Iowa: mean [95% range]  Iowa: n_years  Iowa: years_above_chance  Iowa: range_above_chance
test                                      score         metric                                                                                                 
real: drought years                       penalty_score spearman        +0.16 [-0.01, +0.35]              7                         5                     False
                                                        top_third_hit   +0.49 [+0.42, +0.54]              7                         6                      True
                                          drought_freq  spearman        -0.19 [-0.53, +0.15]              7                         3                     False
                                                        top_third_hit   +0.36 [+0.19, +0.53]              7                         3                     False
                                          storage       spearman        +0.36 [+0.16, +0.56]              7                         6                      True
                                                        top_third_hit   +0.53 [+0.44, +0.61]              7                         6                      True
placebo: zero-drought years               penalty_score spearman        -0.13 [-0.35, +0.09]              2                         1                     False
                                                        top_third_hit   +0.20 [+0.20, +0.20]              2                         0                     False
                                          drought_freq  spearman        -0.13 [-0.36, +0.10]              2                         1                     False
                                                        top_third_hit   +0.25 [+0.19, +0.31]              2                         0                     False
                                          storage       spearman        -0.01 [-0.04, +0.01]              2                         1                     False
                                                        top_third_hit   +0.30 [+0.27, +0.33]              2                         0                     False
placebo: non-D2+ counties, all test years penalty_score spearman        +0.05 [-0.11, +0.22]              7                         3                     False
                                                        top_third_hit   +0.32 [+0.24, +0.40]              7                         3                     False
                                          drought_freq  spearman        +0.03 [-0.21, +0.27]              7                         4                     False
                                                        top_third_hit   +0.36 [+0.24, +0.49]              7                         3                     False
                                          storage       spearman        -0.02 [-0.16, +0.10]              7                         4                     False
                                                        top_third_hit   +0.28 [+0.19, +0.35]              7                         1                     False

### Pre-registered success rule, applied

Penalty score, real test: top-third hit rate 95% range entirely above 0.33, **and** broad placebo (non-D2+ counties, all test years) ranges for the penalty score include chance on both metrics.

In [6]:
def rule(rk, crop):
    s = rk["summary"]; p = s[(s["crop"] == crop) & (s["score"] == "penalty_score")]
    real_hit = p[(p["test"] == st.REAL_TEST) & (p["metric"] == "top_third_hit")].iloc[0]
    plc = p[p["test"] == st.PLACEBO_ALL].set_index("metric")
    hit_clears = bool(real_hit["_lo"] > st.TOP_SHARE)
    placebo_clean = bool(plc.loc["spearman", "_lo"] <= 0 <= plc.loc["spearman", "_hi"]
                         and plc.loc["top_third_hit", "_lo"] <= st.TOP_SHARE <= plc.loc["top_third_hit", "_hi"])
    return {"real top-third hit rate": real_hit["mean [95% range]"], "n test years": int(real_hit["n_years"]),
            "hit-rate range above 0.33": hit_clears,
            "broad placebo Spearman": plc.loc["spearman", "mean [95% range]"], "broad placebo hit rate": plc.loc["top_third_hit", "mean [95% range]"],
            "broad placebo at chance": placebo_clean, "MEETS RULE": hit_clears and placebo_clean}

rows = {}
for crop_label, crop in CROPS.items():
    rows[("Iowa", crop_label)] = rule(rk_ia, crop)
    rows[("Nebraska dryland", crop_label)] = rule(rk_ne, crop) if (rk_ne is not None and PASS[crop]) else {"MEETS RULE": "not run (gate)"}
verdict = pd.DataFrame(rows).T
display(verdict)
for (state, crop_label), r in rows.items():
    print(f"{state} {crop_label}: {'MEETS the pre-registered rule' if r['MEETS RULE'] is True else ('does NOT meet the rule' if r['MEETS RULE'] is False else r['MEETS RULE'])}")

,,real top-third hit rate,n test years,hit-rate range above 0.33,broad placebo Spearman,broad placebo hit rate,broad placebo at chance,MEETS RULE
Iowa,Corn,"+0.45 [+0.36, +0.54]",5,True,"+0.04 [-0.17, +0.27]","+0.35 [+0.25, +0.46]",True,True
Nebraska dryland,Corn,NaN,NaN,NaN,NaN,NaN,NaN,not run (gate)
Iowa,Soybeans,"+0.49 [+0.42, +0.54]",7,True,"+0.05 [-0.11, +0.22]","+0.32 [+0.24, +0.40]",True,True
Nebraska dryland,Soybeans,NaN,NaN,NaN,NaN,NaN,NaN,not run (gate)


Iowa Corn: MEETS the pre-registered rule
Nebraska dryland Corn: not run (gate)
Iowa Soybeans: MEETS the pre-registered rule
Nebraska dryland Soybeans: not run (gate)


## Step 3: plain-language summary

**Outcome: the replication could not be run. Both crops stopped at the pre-registered feasibility gate, so the Nebraska test was never computed and no result exists to judge against the success rule.** The Iowa results above are the reference the method was built on, recomputed with the same functions; they are not new evidence.

**Why the gate stopped each crop.** The rule required at least 20 dryland counties and at least 3 test years (2016–2025) with 10 or more of them at D2+ with a reported yield.
- *Corn:* 13 dryland counties (fewer than 20) and no test year with 10 or more D2+ dryland counties. In 2016–2025 the most D2+ dryland corn counties in any year was 9 (2022 and 2023). Corn fails both conditions.
- *Soybeans:* 18 dryland counties (fewer than 20), although four test years (2020, 2022, 2023, 2024) had 10 or more D2+ dryland counties. Soybeans fail the county-count condition only.

**What this does and does not say.** It says the Nebraska dryland pool is too small, under the definition fixed in advance, for a held-out test of the kind run in Iowa: most Nebraska counties that report corn or soybeans irrigate more than a fifth of the crop (median mean irrigated share about 0.70 for both crops). It does **not** say the method fails in Nebraska; it was not tested there. It also does not say the method works there. The honest statement is "not testable with these data under the pre-registered rules."

**Dryland status is partly assumed.** The irrigation extension's last Nebraska year is 2018, and its coverage thins before that (166 county-crop rows in 2000, 45 in 2018). Dryland status for 2019–2025 is therefore carried forward from earlier years, not measured. Had the test run, this would have been a caveat on every test year.

**On effect sizes from few years.** Had a crop passed with the minimum of 3 test years and a range clearing chance, the size of the effect would likely have been overstated: estimates selected because they cleared a threshold on few years tend to be too large (Gelman, Hill & Vehtari, *Regression and Other Stories*, §4.4, §16.1). The same caution applies to Iowa's results, which rest on 5 (corn) and 7 (soybeans) test years.

**What would make this testable.** A looser dryland cut-off, a pooled corn-plus-soybeans test, or a lower county threshold would each enlarge the pool, but every one of them is a change made after seeing the data and is therefore not run here. If the office wants a Nebraska answer, the choice should be fixed in a new pre-registration and tested once. Alternatives that do not depend on the small dryland pool: test the method on Nebraska's irrigated counties as a *negative control* (irrigation should blunt drought penalties, so the score should predict less there), or use the irrigated-versus-non-irrigated yields in the irrigation extension directly for 2000–2018.

**Scope kept.** Nebraska was not added to the app, the priority map or `soil_tool.py`; nothing under `data/` or `src/` was modified; the method's parameters were asserted from the module and left unchanged.